# Data Extraction and Analysis

This notebook:
1. Parses the questionnaire (DOCX) to extract questions and conditions
2. Parses the concepts (PPTX) to extract product descriptions
3. Analyzes the ground truth Excel data to understand output format
4. Saves parsed data to JSON for later use

In [1]:
import sys
import os
import json
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent))

from src.parsers.questionnaire_parser import QuestionnaireParser
from src.parsers.concept_parser import ConceptParser
from src.parsers.data_analyzer import DataAnalyzer

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Parse Questionnaire

In [2]:
# Path to questionnaire
questionnaire_path = "../source docs/Board games_questionnaire EN MASTER.docx"

# Parse
parser = QuestionnaireParser(questionnaire_path)
questionnaire = parser.parse()

print(f"✓ Parsed {len(questionnaire.questions)} questions")
print(f"✓ Found {len(questionnaire.dimensions)} dimension variables")
print(f"✓ Found {len(questionnaire.screening_rules)} screening rules")

✓ Parsed 25 questions
✓ Found 5 dimension variables
✓ Found 2 screening rules


In [3]:
# Display sample questions
print("\nSample Questions:")
print("=" * 80)
for q_id in ['S1', 'S2', 'B2', 'B3', 'B6']:
    if q_id in questionnaire.questions:
        q = questionnaire.questions[q_id]
        print(f"\n{q.id}: {q.text}")
        print(f"   Type: {q.question_type.value}")
        print(f"   Scale: {q.scale_type.value if q.scale_type else 'N/A'}")
        print(f"   Options: {len(q.options)}")
        if q.conditions:
            print(f"   ⚠️  Conditional: Depends on {q.conditions[0].source_question}")


Sample Questions:

S1: : Gender
   Type: single_coded
   Scale: N/A
   Options: 0

S2: : Age
   Type: single_coded
   Scale: N/A
   Options: 0

B2: : Priced Purchase Intent
   Type: single_coded
   Scale: N/A
   Options: 0

B3: : Uniqueness
   Type: single_coded
   Scale: N/A
   Options: 0

B6: : Likeability
   Type: single_coded
   Scale: N/A
   Options: 0


In [4]:
# Display dimension variables
print("\nDimension Variables (Persona Attributes):")
print("=" * 80)
for dim in questionnaire.dimensions:
    print(f"\n{dim.name} (Q{dim.question_id})")
    print(f"   Type: {'Demographic' if dim.is_demographic else 'Psychographic'}")
    print(f"   Quota: {dim.is_quota}")
    print(f"   Values: {dim.values[:3]}{'...' if len(dim.values) > 3 else ''}")


Dimension Variables (Persona Attributes):

gender (QS1)
   Type: Demographic
   Quota: True
   Values: []

age (QS2)
   Type: Demographic
   Quota: True
   Values: []

category_buyer (QS4)
   Type: Psychographic
   Quota: False
   Values: []

category_non_rejector (QS5)
   Type: Psychographic
   Quota: False
   Values: []

sc_players (QS8)
   Type: Psychographic
   Quota: False
   Values: []


In [5]:
# Save to JSON
output_dir = Path("../data/parsed")
output_dir.mkdir(parents=True, exist_ok=True)

questionnaire_dict = parser.to_dict(questionnaire)
with open(output_dir / "questionnaire.json", 'w', encoding='utf-8') as f:
    json.dump(questionnaire_dict, f, indent=2, ensure_ascii=False)

print("\n✓ Saved questionnaire to data/parsed/questionnaire.json")


✓ Saved questionnaire to data/parsed/questionnaire.json


## 2. Parse Concepts

In [6]:
# Path to concepts
concepts_path = "../source docs/Tech_Enabled_CZ_Concepts_EN.pptx"

# Parse
concept_parser = ConceptParser(concepts_path)
concepts = concept_parser.parse()

print(f"✓ Parsed {len(concepts)} concepts")

✓ Parsed 8 concepts


In [7]:
# Display concepts
print("\nProduct Concepts:")
print("=" * 80)
for i, concept in enumerate(concepts, 1):
    print(f"\n{i}. {concept.name}")
    print(f"   Price: {concept.price}")
    print(f"   Occasion: {concept.occasion}")
    print(f"   Features: {len(concept.features)}")
    print(f"\n   Description for LLM:")
    print(f"   {concept.description[:200]}...")


Product Concepts:

1. Concept 1: Christmas AR Message
   Price: Imagine that a new personalized video scratch card has just been launched!

🎁 This Christmas gift scratch card is sold for 250 CZK and guarantees a win for the recipient — making it an ideal gift under the Christmas tree.

On the front side of the card, there is a QR code that can be scanned with a phone. After scanning, you can record a short video where you wish the recipient a Merry Christmas. A festive filter with a holiday design 🎄 is automatically added to the video.

When the recipient scratches the card, discovers their prize, and scans the QR code, your personal greeting will appear directly above the card — brought to life through augmented reality and fun digital effects.

Available at all sales points where scratch cards are sold, or online through relevant websites and apps.
   Occasion: Christmas
   Features: 0

   Description for LLM:
   Concept 1: Christmas AR Message
Price: Imagine that a new personalized

In [8]:
# Save to JSON
concepts_dict = concept_parser.to_dict(concepts)
with open(output_dir / "concepts.json", 'w', encoding='utf-8') as f:
    json.dump(concepts_dict, f, indent=2, ensure_ascii=False)

print("\n✓ Saved concepts to data/parsed/concepts.json")


✓ Saved concepts to data/parsed/concepts.json


## 3. Analyze Ground Truth Data

In [9]:
# Path to ground truth data
data_path = "../source docs/KAP400232611_Respondent_Data_Tech Enabled CZ.xlsx"

# Analyze
analyzer = DataAnalyzer(data_path)
structure = analyzer.analyze()

print(f"✓ Analyzed Excel file")
print(f"   Rows: {structure.num_rows}")
print(f"   Columns: {structure.num_columns}")

✓ Analyzed Excel file
   Rows: 400
   Columns: 227


In [10]:
# Display column structure
print("\nColumn Categories:")
print("=" * 80)
print(f"\nDemographic columns ({len(structure.demographic_columns)}):")
print(structure.demographic_columns[:10])

print(f"\nID columns ({len(structure.id_columns)}):")
print(structure.id_columns)

print(f"\nQuestion columns by concept:")
for concept, cols in list(structure.question_columns.items())[:2]:
    print(f"\n{concept}: {len(cols)} columns")
    print(f"   {cols[:5]}...")


Column Categories:

Demographic columns (85):
['SERIAL', 'DATE', 'Gender', '(SEX) SEX', 'AGE', '(GROUPFMR) SAMPLE TYPE', 'Dummy : Position - 2 CZ Christmas AR Message  ', 'Dummy : Position - 2 CZ Birthday Message  ', 'Dummy : Position - 2 CZ Valentine Message  ', '(PRPURINT) PRICED PURCHASE INTENT - 2 CZ Christmas AR Message  ']

ID columns (19):
['SERIAL', 'Concept ID - 2 CZ Christmas AR Message  ', 'Concept ID - 2 CZ Elf Yourself  ', 'Concept ID - 2 CZ AR Christmas Mini Game  ', 'Concept ID - 2 CZ Christmas Lip Sync  ', 'Concept ID - 2 CZ Christmas Duet  ', 'Concept ID - 2 CZ Christmas Cake Suprise  ', 'Concept ID - 2 CZ Birthday Message  ', 'Concept ID - 2 CZ Valentine Message  ', 'Parent ID - 2 CZ Christmas AR Message  ', 'Parent ID - 2 CZ Elf Yourself  ', 'Parent ID - 2 CZ AR Christmas Mini Game  ', 'Parent ID - 2 CZ Christmas Lip Sync  ', 'Parent ID - 2 CZ Christmas Duet  ', 'Parent ID - 2 CZ Christmas Cake Suprise  ', 'Parent ID - 2 CZ Birthday Message  ', 'Parent ID - 2 CZ Val

In [11]:
# Examine a specific question column format
print("\nExample: Purchase Intent Column Format")
print("=" * 80)

# Find a purchase intent column
pi_cols = [col for col in analyzer.df.columns if 'PRPURINT' in col]
if pi_cols:
    col_format = analyzer.get_column_format(pi_cols[0])
    print(f"\nColumn: {col_format['column_name']}")
    print(f"Type: {col_format['dtype']}")
    print(f"Has code format: {col_format['has_code_format']}")
    print(f"\nSample values:")
    for val in col_format['sample_values'][:5]:
        print(f"   {val}")


Example: Purchase Intent Column Format

Column: (PRPURINT) PRICED PURCHASE INTENT - 2 CZ Christmas AR Message  
Type: object
Has code format: True

Sample values:
   (3) Might or might not
   (2) Probably would not
   (4) Probably would
   (3) Might or might not
   (3) Might or might not


In [12]:
# Save structure to JSON
structure_dict = analyzer.to_dict(structure)
with open(output_dir / "data_structure.json", 'w', encoding='utf-8') as f:
    json.dump(structure_dict, f, indent=2, ensure_ascii=False)

print("\n✓ Saved data structure to data/parsed/data_structure.json")


✓ Saved data structure to data/parsed/data_structure.json


## 4. Summary

We have successfully:
- ✅ Parsed the questionnaire and extracted all questions, scales, and conditions
- ✅ Extracted all 8 product concepts with descriptions
- ✅ Analyzed the ground truth data structure
- ✅ Saved all parsed data to JSON files

**Next Steps:**
- Move to notebook 02 to set up scales and test SSR
- Review and refine anchor statements for each scale